In [1]:
#pip install pyarrow
#pip install kagglehub
#pip install spark

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arvindnagaonkar/flight-delay")

print("Path to dataset files:", path)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/codespace/.cache/kagglehub/datasets/arvindnagaonkar/flight-delay/versions/2


# We Can Explore with Pandas Dataframes, though this takes up more space in our virtual environment (Github Codespaces)

In [3]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

# List files in the dataset directory to find the CSV file
files = os.listdir(path)
print("Files in dataset directory:", files)

data_files = [f for f in files if f.endswith('.parquet')] 
if data_files: 
    data_file_path = os.path.join(path, data_files[0])

    schema = pq.read_schema(data_file_path)
    print("Available columns:")
    print(schema.names)

    cols_to_read = ['FlightDate', 'OriginCityName', 'DestCityName']  # columns grabbed

    table = pq.read_table(
        data_file_path,
        columns=cols_to_read
    )

    # Then slice just a few rows
    #   without this my github codespace instance crashes
    small_df = table.slice(0, 1000).to_pandas()
    print(small_df.head())
else:
    print("No data file found in the dataset directory.")

Files in dataset directory: ['features_added.parquet', 'Flight_Delay.parquet']
Available columns:
['Year', 'Month', 'DayofMonth', 'FlightDate', 'Marketing_Airline_Network', 'OriginCityName', 'DestCityName', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'TaxiOut', 'TaxiIn', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'DayofWeek', 'Holidays', 'CRSDepTimeMinute', 'CRSDepTimeHour', 'WheelsOffMinute', 'WheelsOffHour', 'CRSArrTimeMinute', 'CRSArrTimeHour', 'WheelsOnMinute', 'WheelsOnHour', 'CRSDepTimeHourDis', 'WheelsOffHourDis', 'CRSArrTimeHourDis', 'WheelsOnHourDis', 'CRSElapsedTimeGorup', '__index_level_0__']
  FlightDate OriginCityName    DestCityName
0 2018-01-15     Newark, NJ  Charleston, SC
1 2018-01-16     Newark, NJ  Charleston, SC
2 2018-01-17     Newark, NJ  Charleston, SC
3 2018-01-18     Newark, NJ  Charleston, SC
4 2018-01-2

# We will do the rest of the exploration with Spark

In [4]:
from pyspark.sql import SparkSession

# Start Spark session
spark = SparkSession.builder \
    .appName("FlightDelayAnalysis") \
    .getOrCreate()

# Get our parquet file path
files = os.listdir(path)
print("Files in dataset directory:", files)
data_files = [f for f in files if f.endswith('.parquet')] 
data_file_path = os.path.join(path, data_files[0])
df = spark.read.parquet(data_file_path)

# Show schema (column names and types)
df.printSchema()
df.show(10)

# Select specific columns and filter
df.select("FlightDate", "OriginCityName", "DestCityName").show(5)

print("Total rows:", df.count())

# Run basic aggregation
df.groupBy("OriginCityName").count().orderBy("count", ascending=False).show(10)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/07 13:46:26 WARN Utils: Your hostname, codespaces-90a6fa, resolves to a loopback address: 127.0.0.1; using 10.0.13.150 instead (on interface eth0)
25/10/07 13:46:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/07 13:46:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Files in dataset directory: ['features_added.parquet', 'Flight_Delay.parquet']


root
 |-- Year: long (nullable = true)
 |-- Month: long (nullable = true)
 |-- DayofMonth: long (nullable = true)
 |-- FlightDate: timestamp_ntz (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- DepTime: double (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- DepDelayMinutes: double (nullable = true)
 |-- TaxiOut: double (nullable = true)
 |-- TaxiIn: double (nullable = true)
 |-- ArrTime: double (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- ArrDelayMinutes: double (nullable = true)
 |-- CRSElapsedTime: double (nullable = true)
 |-- ActualElapsedTime: double (nullable = true)
 |-- AirTime: double (nullable = true)
 |-- Distance: double (nullable = true)
 |-- DistanceGroup: long (nullable = true)
 |-- CarrierDelay: double (nullable = true)
 |-- WeatherDelay: double (nullable = true)
 |-- NASDelay: double (nullable = true)
 |-- SecurityDel

25/10/07 13:46:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----+-----+----------+-------------------+-------------------------+--------------+--------------+-------+--------+---------------+-------+------+-------+--------+---------------+--------------+-----------------+-------+--------+-------------+------------+------------+--------+-------------+-----------------+---------+--------+----------------+--------------+---------------+-------------+----------------+--------------+--------------+------------+-----------------+----------------+-----------------+---------------+-------------------+-----------------+
|Year|Month|DayofMonth|         FlightDate|Marketing_Airline_Network|OriginCityName|  DestCityName|DepTime|DepDelay|DepDelayMinutes|TaxiOut|TaxiIn|ArrTime|ArrDelay|ArrDelayMinutes|CRSElapsedTime|ActualElapsedTime|AirTime|Distance|DistanceGroup|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|DayofWeek|Holidays|CRSDepTimeMinute|CRSDepTimeHour|WheelsOffMinute|WheelsOffHour|CRSArrTimeMinute|CRSArrTimeHour|WheelsOnMinute|W

+-------------------+--------------+--------------+
|         FlightDate|OriginCityName|  DestCityName|
+-------------------+--------------+--------------+
|2018-01-15 00:00:00|    Newark, NJ|Charleston, SC|
|2018-01-16 00:00:00|    Newark, NJ|Charleston, SC|
|2018-01-17 00:00:00|    Newark, NJ|Charleston, SC|
|2018-01-18 00:00:00|    Newark, NJ|Charleston, SC|
|2018-01-20 00:00:00|    Newark, NJ|Charleston, SC|
+-------------------+--------------+--------------+
only showing top 5 rows
Total rows: 30132631


+--------------------+-------+
|      OriginCityName|  count|
+--------------------+-------+
|         Chicago, IL|1708956|
|         Atlanta, GA|1477844|
|Dallas/Fort Worth...|1166054|
|          Denver, CO|1131732|
|        New York, NY|1106836|
|       Charlotte, NC| 966926|
|         Houston, TX| 894020|
|     Los Angeles, CA| 877500|
|      Washington, DC| 875581|
|         Seattle, WA| 731920|
+--------------------+-------+
only showing top 10 rows


## North Carolina Exploration

In [ ]:
from pyspark.sql.functions import count, avg

nc_flights = df.filter(
    (df.OriginCityName.contains("NC")) | (df.DestCityName.contains("NC"))
)

# Count flights grouped by origin and destination city
#     Also calculate average flight duration
#     Order by flight count descending
result = nc_flights.groupBy("OriginCityName", "DestCityName") \
    .agg(
        count("*").alias("flight_count"),
        avg("ActualElapsedTime").alias("avg_flight_duration")
    ) \
    .orderBy("flight_count", ascending=False)
result.show(20)


+------------------+------------------+------------+-------------------+
|    OriginCityName|      DestCityName|flight_count|avg_flight_duration|
+------------------+------------------+------------+-------------------+
|      New York, NY|     Charlotte, NC|       36644|  120.4040770658225|
|     Charlotte, NC|      New York, NY|       35312| 110.22666515632079|
|      New York, NY|Raleigh/Durham, NC|       31054| 104.18235976041734|
|Raleigh/Durham, NC|      New York, NY|       30764|  97.22734364842023|
|    Washington, DC|     Charlotte, NC|       25985|  88.46238214354435|
|     Charlotte, NC|    Washington, DC|       25316|   79.7194659503871|
|     Charlotte, NC|       Atlanta, GA|       24692|  70.83962416977158|
|       Atlanta, GA|     Charlotte, NC|       24316|  70.11584964632341|
|       Chicago, IL|     Charlotte, NC|       21836| 112.65456127495878|
|     Charlotte, NC|       Chicago, IL|       21723|  123.1069833816692|
|Raleigh/Durham, NC|       Atlanta, GA|       19449

In [ ]:
# From NC
nc_origin = df.filter(df.OriginCityName.contains("NC"))

# Count by origin city
#   Order by count descending
nc_origin.groupBy("OriginCityName").count().orderBy("count", ascending=False).show(10)


+--------------------+------+
|      OriginCityName| count|
+--------------------+------+
|       Charlotte, NC|966926|
|  Raleigh/Durham, NC|240054|
|Greensboro/High P...| 62010|
|       Asheville, NC| 41338|
|      Wilmington, NC| 34389|
|    Fayetteville, NC| 15762|
|Jacksonville/Camp...| 13244|
|New Bern/Morehead...|  8390|
|      Greenville, NC|  4600|
|         Concord, NC|  3456|
+--------------------+------+
only showing top 10 rows


In [ ]:
nc_flights = df.filter(
    (df.OriginCityName.contains("NC")) | (df.DestCityName.contains("NC"))
)

# Average arrival delay grouped by origin city
#  Order by average delay descending
nc_flights.groupBy("OriginCityName") \
    .agg(avg("ArrDelayMinutes").alias("AvgArrDelay")) \
    .orderBy("AvgArrDelay", ascending=False).show(10)

+------------------+------------------+
|    OriginCityName|       AvgArrDelay|
+------------------+------------------+
| State College, PA|              52.2|
|Montrose/Delta, CO|              34.0|
|          Reno, NV|              33.5|
|       Bozeman, MT|30.663414634146342|
|       Jackson, WY| 30.24074074074074|
|       Concord, NC| 27.73582175925926|
|       Ontario, CA|  26.2311320754717|
|       Trenton, NJ| 22.61545711592837|
|       CONCORD, NC|21.515097690941385|
|     Lafayette, LA|20.787790697674417|
+------------------+------------------+
only showing top 10 rows


In [ ]:
# Most popular months for flights to/from NC
nc_flights.groupBy("Year", "Month", "OriginCityName") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20)

+----+-----+--------------+-----+
|Year|Month|OriginCityName|count|
+----+-----+--------------+-----+
|2019|   10| Charlotte, NC|18156|
|2019|    8| Charlotte, NC|17875|
|2019|    9| Charlotte, NC|17852|
|2019|   11| Charlotte, NC|17680|
|2019|    7| Charlotte, NC|17679|
|2020|    1| Charlotte, NC|17651|
|2019|    5| Charlotte, NC|17586|
|2021|    5| Charlotte, NC|17541|
|2019|   12| Charlotte, NC|17419|
|2019|    3| Charlotte, NC|17252|
|2019|    4| Charlotte, NC|17099|
|2018|    8| Charlotte, NC|17013|
|2018|   10| Charlotte, NC|16899|
|2019|    6| Charlotte, NC|16773|
|2021|    8| Charlotte, NC|16715|
|2018|    5| Charlotte, NC|16661|
|2021|    4| Charlotte, NC|16654|
|2021|    7| Charlotte, NC|16637|
|2021|   10| Charlotte, NC|16581|
|2019|    1| Charlotte, NC|16560|
+----+-----+--------------+-----+
only showing top 20 rows


In [13]:
# Percentage of delayed flights (>15 min) from Charlotte, NC by month and year
#   Now that we know the most popular months for NC flights, let's see the delay percentage
from pyspark.sql.functions import when, col, sum as spark_sum

delay_threshold = 15

nc_delays = nc_flights.filter(nc_flights.OriginCityName == "Charlotte, NC") \
    .withColumn("is_delayed", when(col("ArrDelayMinutes") > delay_threshold, 1).otherwise(0)) \
    .groupBy("Year", "Month") \
    .agg(
        count("*").alias("total_flights"),
        spark_sum("is_delayed").alias("delayed_flights")
    ) \
    .withColumn("delay_percentage", col("delayed_flights") / col("total_flights") * 100) \
    .orderBy("delay_percentage", ascending=False)

nc_delays.show(20)

+----+-----+-------------+---------------+------------------+
|Year|Month|total_flights|delayed_flights|  delay_percentage|
+----+-----+-------------+---------------+------------------+
|2022|    7|        15286|           5758| 37.66845479523747|
|2022|    6|        14800|           5144| 34.75675675675676|
|2018|    7|        16426|           5595| 34.06185315962498|
|2018|    6|        15304|           5158| 33.70360690015682|
|2018|    8|        17013|           5509| 32.38112031975548|
|2019|    6|        16773|           5386|32.111130984320035|
|2019|    8|        17875|           5300| 29.65034965034965|
|2020|    2|        16388|           4784|29.192091774469127|
|2022|   12|        13863|           4028|29.055759936521675|
|2019|    4|        17099|           4952|28.960757939060766|
|2023|    4|        14727|           4262|28.940042099545053|
|2018|    5|        16661|           4785|28.719764720004804|
|2021|    7|        16637|           4687|28.172146420628717|
|2019|  

In [14]:
# It might be easier to read if we group by seasons instead of months
#   And find the most popular seasons for flights from Charlotte, NC

from pyspark.sql.functions import expr

season_df = nc_flights.filter(nc_flights.OriginCityName == "Charlotte, NC") \
    .withColumn(
        "Season",
        expr("""
            CASE 
                WHEN Month IN (12, 1, 2) THEN 'Winter'
                WHEN Month IN (3, 4, 5) THEN 'Spring'
                WHEN Month IN (6, 7, 8) THEN 'Summer'
                WHEN Month IN (9, 10, 11) THEN 'Fall'
            END
        """)
    ) \
    .groupBy("Year", "Season") \
    .count() \
    .orderBy("count", ascending=False)

season_df.show()

+----+------+-----+
|Year|Season|count|
+----+------+-----+
|2019|  Fall|53688|
|2019|Summer|52327|
|2019|Spring|51937|
|2021|Spring|50715|
|2021|Summer|49685|
|2018|  Fall|49167|
|2019|Winter|49043|
|2018|Summer|48743|
|2018|Spring|48212|
|2021|  Fall|48032|
|2020|Winter|46537|
|2022|Summer|45593|
|2018|Winter|44791|
|2022|  Fall|44130|
|2022|Spring|43828|
|2022|Winter|42282|
|2021|Winter|39342|
|2020|  Fall|36976|
|2020|Summer|33104|
|2020|Spring|32346|
+----+------+-----+
only showing top 20 rows
